## Calculating Venus "OR/Target" attitudes from given PCAD/ACA attitudes

In [9]:
from astropy.table import Table
from Quaternion import Quat
import chandra_aca

These are the attitudes from Eric's original '2017_Venus_Obs_Summary.txt' with the new obsids used as the labels and other columns cut.

In [2]:
PCAD_atts = """
Obs     Start_Time    End_Time        q1          q2           q3           q4
17439   009:18:21:00  009:20:05:30    -0.54124474  0.17440748  -0.10503936  0.81584490
18690   009:20:09:30  009:21:54:00    -0.54128440  0.17382505  -0.10475117  0.81597993
18691   009:21:58:00  009:23:42:30    -0.54133029  0.17320695  -0.10443884  0.81612095
18692   009:23:46:30  010:01:31:00    -0.54131932  0.17257327  -0.10423103  0.81628902
18693   010:01:35:00  010:03:19:30    -0.54134818  0.17195804  -0.10395069  0.81643544
18694   010:03:23:30  010:05:08:00    -0.54136894  0.17131557  -0.10367493  0.81659179
18695   010:05:12:00  010:06:56:30    -0.54137633  0.17071488  -0.10344604  0.81674171
18696   010:07:00:30  010:08:45:00    -0.54140053  0.17002964  -0.10314792  0.81690630
"""

In [3]:
t = Table.read(PCAD_atts, format='ascii')

This is the "original"/"used for most of the mission" ODB_SI_ALIGN transform matrix.  I think it is also the reference matrix being used in SAUSAGE.  However, OFLS is using the 07JUL16 characteristics version, so I'm not 100% that this is the right matrix.  However, if I take the PCAD attitudes in pre-products, and invert them by this matrix, I end up with the current OR-list attitudes (off by the dynamic offsets).

In [15]:
ODB_SI_ALIGN = chandra_aca.transform.ODB_SI_ALIGN

In [16]:
ODB_SI_ALIGN

array([[  9.99999906e-01,  -3.37419984e-04,  -2.73439987e-04],
       [  3.37419984e-04,   9.99999943e-01,  -4.61320600e-08],
       [  2.73439987e-04,  -4.61320600e-08,   9.99999963e-01]])

Here, for each of these obsids, I've converted from ACA to "target" coordinates for each, added each attitude to a data structure and then just printed the thing.  The full code for calc_targ_from_aca is just the multiplication in: https://github.com/sot/chandra_aca/blob/ca0442f35b35da208d9ddb64f4db911e95a42ea6/chandra_aca/transform.py#L159

In [12]:
targ_atts = []
for obs in t:
    q_aca = Quat([obs['q1'], obs['q2'], obs['q3'], obs['q4']])
    q_targ = chandra_aca.calc_targ_from_aca(q_aca, y_off=0, z_off=0, si_align=ODB_SI_ALIGN)
    targ_atts.append({'obsid': obs['Obs'],
                     'q1': q_targ.q[0], 'q2': q_targ.q[1], 'q3': q_targ.q[2], 'q4': q_targ.q[3],
                     'ra': q_targ.ra, 'dec': q_targ.dec, 'roll': q_targ.roll})
targ_atts = Table(targ_atts)

In [14]:
targ_atts[['obsid', 'q1', 'q2', 'q3', 'q4', 'ra', 'dec', 'roll']]

obsid,q1,q2,q3,q4,ra,dec,roll
int64,float64,float64,float64,float64,float64,float64,float64
17439,-0.541229663934,0.174387246974,-0.104827717354,0.815886446944,338.579685658,-9.85112103621,291.014641357
18690,-0.541269382794,0.173804805217,-0.104539499158,0.816021348691,338.648173371,-9.81624028852,291.032228435
18691,-0.541315334371,0.173186693694,-0.1042271391,0.816162231488,338.721400868,-9.77952690438,291.050314569
18692,-0.541304442867,0.172552988879,-0.104019302249,0.816330179787,338.786324325,-9.7359722092,291.076198486
18693,-0.541333368334,0.171937743744,-0.103738933608,0.816476468373,338.856403186,-9.69780114657,291.096184493
18694,-0.541354199023,0.171295255886,-0.103463144398,0.816632684008,338.928024491,-9.65703625386,291.118169435
18695,-0.541361659071,0.170694546649,-0.1032342281,0.816782483262,338.99231415,-9.61730422493,291.140155315
18696,-0.541385933918,0.170009288246,-0.102936077031,0.816946929276,339.068981826,-9.57397355144,291.163139357


## If we want to pre-correct for dynamic offsets, we can use the offsets in the pre-products dynamic offset table. 

It also looks like all of these dynamic offsets are small.  I think these would be larger if the observations were handled as cycle 18 observations (presently being handled as cycle 16 observations, which seems fine).

In [17]:
dynamic_offsets = Table.read("""
obsid detector chipx chipy chip_id target_offset_y target_offset_z target_ra target_dec target_roll aca_offset_y aca_offset_z aca_ra aca_dec aca_roll mean_date mean_t_ccd
17764  ACIS-S  200.7  476.9  7  0.00  -18.00  76.956542  30.401439  262.690000  7.20  0.72  76.947191  30.423980  262.694735  2017:009:12:20:15.816  -13.25
17439  ACIS-I  932.0  1009.0  3  0.00  0.00  338.557806  -9.838692  291.010900  6.69  -0.71  338.535439  -9.824457  291.007081  2017:009:19:10:15.816  -13.58
18690  ACIS-I  932.0  1009.0  3  0.00  0.00  338.626292  -9.803817  291.028500  7.00  -0.56  338.603851  -9.789523  291.024682  2017:009:20:59:35.816  -13.66
18691  ACIS-I  932.0  1009.0  3  0.00  0.00  338.699513  -9.767111  291.046600  7.21  -0.45  338.677022  -9.752780  291.042788  2017:009:22:48:55.816  -13.71
18692  ACIS-I  932.0  1009.0  3  0.00  0.00  338.764439  -9.723566  291.072500  7.34  -0.39  338.741913  -9.709217  291.068699  2017:010:00:38:15.816  -13.75
18693  ACIS-I  932.0  1009.0  3  0.00  0.00  338.834516  -9.685402  291.092500  7.42  -0.35  338.811971  -9.671045  291.088710  2017:010:02:27:35.816  -13.77
18694  ACIS-I  932.0  1009.0  3  0.00  0.00  338.906135  -9.644646  291.114500  7.44  -0.34  338.883580  -9.630291  291.110724  2017:010:04:16:55.816  -13.77
18695  ACIS-I  932.0  1009.0  3  0.00  0.00  338.970422  -9.604922  291.136500  7.44  -0.34  338.947866  -9.590577  291.132740  2017:010:06:03:31.816  -13.77
18696  ACIS-I  932.0  1009.0  3  0.00  0.00  339.047088  -9.561600  291.159500  7.41  -0.36  339.024536  -9.547271  291.155757  2017:010:07:44:39.816  -13.76""", format='ascii')

In [21]:
targ_atts_dyn = []
for obs in t:
    q_aca = Quat([obs['q1'], obs['q2'], obs['q3'], obs['q4']])
    dyn = dynamic_offsets[dynamic_offsets['obsid'] == obs['Obs']][0]
    # calc_targ_from_aca takes the offsets in degrees with the OR-list sign convention
    q_targ = chandra_aca.calc_targ_from_aca(q_aca,
                                            y_off=(dyn['aca_offset_y'] / 3600.),
                                            z_off=(dyn['aca_offset_z'] / 3600.), si_align=ODB_SI_ALIGN)
    targ_atts_dyn.append({'obsid': obs['Obs'],
                     'q1': q_targ.q[0], 'q2': q_targ.q[1], 'q3': q_targ.q[2], 'q4': q_targ.q[3],
                     'ra': q_targ.ra, 'dec': q_targ.dec, 'roll': q_targ.roll})
targ_atts_dyn = Table(targ_atts_dyn)

In [23]:
targ_atts_dyn[['obsid', 'q1', 'q2', 'q3', 'q4', 'ra', 'dec', 'roll']]

obsid,q1,q2,q3,q4,ra,dec,roll
int64,float64,float64,float64,float64,float64,float64,float64
17439,-0.541226655426,0.174397428297,-0.104815417595,0.815887846678,338.580175184,-9.85292649443,291.014725114
18690,-0.541266291621,0.173815097444,-0.104526387259,0.8160228865,338.648734241,-9.8181110189,291.032324063
18691,-0.541312193733,0.173197044821,-0.104213465061,0.816163864068,338.722012474,-9.78144092544,291.050418462
18692,-0.54130127428,0.172563391848,-0.104005289285,0.8163318673,338.786965683,-9.73791365964,291.076306954
18693,-0.541330187671,0.171948173176,-0.103724707243,0.816478188266,338.857063789,-9.69975910988,291.096295782
18694,-0.541351024352,0.171305692274,-0.103448862527,0.816634408653,338.928690435,-9.65899814879,291.118281156
18695,-0.541358495423,0.170704983296,-0.103219943534,0.816784204274,338.992980779,-9.61926586782,291.140266696
18696,-0.541382790249,0.170019725702,-0.102921875176,0.816948629748,339.069640863,-9.57592916493,291.163248978
